# Data Cleaning
This notebook handles data cleaning, handling missing variables, trimming features, and renaming them for consistency.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/globalterrorismdb_0718dist.csv'
df = pd.read_csv(RAW_PATH, encoding='latin-1', low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 181,691 rows × 135 columns


In [2]:
# Drop columns where >60% values are null
threshold = 0.6 * len(df)
df = df.dropna(thresh=threshold, axis=1)

# Keep only key columns
keep_cols = ['eventid', 'iyear', 'country_txt', 'region_txt', 'attacktype1_txt', 
             'targtype1_txt', 'gname', 'nkill', 'nwound']
available_cols = [c for c in keep_cols if c in df.columns]
df = df[available_cols].copy()

# Rename columns
rename_map = {
    'eventid': 'event_id',
    'iyear': 'year',
    'country_txt': 'country',
    'region_txt': 'region',
    'attacktype1_txt': 'attack_type',
    'targtype1_txt': 'target_type',
    'gname': 'group_name',
    'nkill': 'killed',
    'nwound': 'wounded'
}
df.rename(columns=rename_map, inplace=True)
print(f'Shape after selection: {df.shape}')

Shape after selection: (181691, 9)


In [3]:
req_cols = ['year', 'country', 'region', 'attack_type']
existing_req = [c for c in req_cols if c in df.columns]
df.dropna(subset=existing_req, inplace=True)

if 'killed' in df.columns:
    df['killed'].fillna(0, inplace=True)
if 'wounded' in df.columns:
    df['wounded'].fillna(0, inplace=True)
    
if 'killed' in df.columns and 'wounded' in df.columns:
    df['casualties'] = df['killed'] + df['wounded']

print("Null checks passed.")
df.head()

Null checks passed.


,event_id,year,country,region,attack_type,target_type,group_name,killed,wounded,casualties
0,197000000001,1970,Dominican Republic,Central America & Caribbean,Assassination,Private Citizens & Property,MANO-D,1.0,0.0,1.0
1,197000000002,1970,Mexico,North America,Hostage Taking (Kidnapping),Government (Diplomatic),23rd of September Communist League,0.0,0.0,0.0
2,197001000001,1970,Philippines,Southeast Asia,Assassination,Journalists & Media,Unknown,1.0,0.0,1.0
3,197001000002,1970,Greece,Western Europe,Bombing/Explosion,Government (Diplomatic),Unknown,NaN,NaN,NaN
4,197001000003,1970,Japan,East Asia,Facility/Infrastructure Attack,Government (Diplomatic),Unknown,NaN,NaN,NaN


In [4]:
import os
OUT_PATH = '../data/processed/gtd_cleaned.csv'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f'Saved cleaned data to {OUT_PATH}')

Saved cleaned data to ../data/processed/gtd_cleaned.csv
